In [1]:
import heapq
import copy

class PuzzleNode:
    def __init__(self, state, parent=None, move="", depth=0):
        self.state = state
        self.parent = parent
        self.move = move
        self.depth = depth
        self.h = self._calculate_manhattan_distance()
        self.f = self.depth + self.h

    def _calculate_manhattan_distance(self):
        distance = 0
        for i in range(4):
            for j in range(4):
                val = self.state[i][j]
                if val != 0:
                    target_row = (val - 1) // 4
                    target_col = (val - 1) % 4
                    distance += abs(i - target_row) + abs(j - target_col)
        return distance

    def get_successors(self):
        successors = []
        empty_row, empty_col = -1, -1

        for i in range(4):
            for j in range(4):
                if self.state[i][j] == 0:
                    empty_row, empty_col = i, j
                    break

        directions = {
            "Lên": (-1, 0),
            "Xuống": (1, 0),
            "Trái": (0, -1),
            "Phải": (0, 1)
        }

        for move_name, (dr, dc) in directions.items():
            new_row, new_col = empty_row + dr, empty_col + dc

            if 0 <= new_row < 4 and 0 <= new_col < 4:
                new_state = copy.deepcopy(self.state)
                new_state[empty_row][empty_col], new_state[new_row][new_col] = \
                    new_state[new_row][new_col], new_state[empty_row][empty_col]

                successors.append(PuzzleNode(new_state, self, move_name, self.depth + 1))

        return successors

    def is_goal(self):
        return self.h == 0

    def __lt__(self, other):
        return self.f < other.f

    def print_board(self):
        for row in self.state:
            print('\t'.join([str(x) if x != 0 else '_' for x in row]))
        print(f"g(n) = {self.depth}, h(n) = {self.h}, f(n) = {self.f}\n")


def solve_15_puzzle(initial_state):
    initial_node = PuzzleNode(initial_state)

    if initial_node.is_goal():
        return [initial_node]

    open_list = []
    heapq.heappush(open_list, initial_node)

    closed_set = set()

    while open_list:
        current_node = heapq.heappop(open_list)

        state_tuple = tuple(tuple(row) for row in current_node.state)

        if state_tuple in closed_set:
            continue

        closed_set.add(state_tuple)

        if current_node.is_goal():
            path = []
            while current_node:
                path.append(current_node)
                current_node = current_node.parent
            return path[::-1]

        for successor in current_node.get_successors():
            succ_tuple = tuple(tuple(row) for row in successor.state)
            if succ_tuple not in closed_set:
                heapq.heappush(open_list, successor)

    return None


if __name__ == "__main__":
    start_board = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 0, 12],
        [13, 14, 11, 15]
    ]

    print("Đang phân tích đường đi...")
    solution_path = solve_15_puzzle(start_board)

    if solution_path:
        print(f"Đã tìm thấy lời giải trong {len(solution_path) - 1} bước:\n")
        for step, node in enumerate(solution_path):
            if step == 0:
                print("--- TRẠNG THÁI BẮT ĐẦU ---")
            else:
                print(f"--- BƯỚC {step}: Trượt ô '{node.move}' ---")
            node.print_board()
    else:
        print("Không tìm thấy đường đi khả thi (cấu hình bị khóa).")

Đang phân tích đường đi...
Đã tìm thấy lời giải trong 2 bước:

--- TRẠNG THÁI BẮT ĐẦU ---
1	2	3	4
5	6	7	8
9	10	_	12
13	14	11	15
g(n) = 0, h(n) = 2, f(n) = 2

--- BƯỚC 1: Trượt ô 'Xuống' ---
1	2	3	4
5	6	7	8
9	10	11	12
13	14	_	15
g(n) = 1, h(n) = 1, f(n) = 2

--- BƯỚC 2: Trượt ô 'Phải' ---
1	2	3	4
5	6	7	8
9	10	11	12
13	14	15	_
g(n) = 2, h(n) = 0, f(n) = 2

